# Fine-tuning YOLO26s pascapruning · Filkom

Mulai dari satu student raw LAMP + DepGraph dari notebook pruning 06. Gunakan konfigurasi supervised baseline YOLO26s 02 dengan optimizer baru dan loss deteksi native. Evaluasi ketiga tahap pada validation yang sama. Catat versi dan SHA-256 raw untuk pairing dengan KD. Dengan input kosong notebook berstatus `NOT_RUN`.

## 01 — Input student dan konfigurasi baseline

In [ ]:
import os
from pathlib import Path

PLATFORM = 'filkom'
WORKSPACE_ROOT = Path(os.environ.get("SKRIPSI_ROOT", Path.cwd())).expanduser()
DATASET_ROOT = WORKSPACE_ROOT / "datasets/poltrada-v3"
OUTPUT_DIR = WORKSPACE_ROOT / "outputs/filkom/fine-tunned/yolo26s"
DATA_YAML = DATASET_ROOT / "data.yaml"
RAW_ARTIFACT_REF = None   # "entity/project/raw-artifact:vN"
RAW_MANIFEST_PATH = None  # Path(".../raw_pruned_manifest.json"), alternatif input lokal
EXPECTED_RAW_SHA256 = None  # Opsional, SHA-256 raw dari student lock untuk pairing KD
INSTALL_DEPENDENCIES = True
WANDB_TRACKING = True
MODE = "full_val"  # "smoke" untuk pemeriksaan satu epoch, AMP nonaktif dan subset val

# Protokol dari baseline 02. Pilih satu SEED aktif dari 42, 0, 2024, 7, 3407.
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0
WORKERS = 2
OPTIMIZER = 'AdamW'
LR = 0.001
PATIENCE = 100
AMP = True
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = 'val'
WANDB_ENTITY = 'ilhamrafiqin-brawijaya-university'
WANDB_PROJECT = 'yolom26-kd-indotraffic'

In [ ]:
def fine_tuned_train_args(DATA, PROJECT, RUN_NAME):
    args = dict(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE, weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC, project=PROJECT, name=RUN_NAME, exist_ok=False, workers=WORKERS, patience=PATIENCE, plots=True, amp=AMP, pretrained=True, save=True)
    if MODE == "smoke":
        args.update(epochs=1, amp=False, nbs=BATCH)
    return args


def fine_tuned_val_args(DATA):
    return dict(data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True, verbose=False)

## 02 — Kernel, verifikasi checkpoint, dan fingerprint train/val

In [ ]:
if RAW_ARTIFACT_REF or RAW_MANIFEST_PATH:
    """Bootstrap declarations embedded in the fine-tuning notebooks."""
    # ruff: noqa: F401, F821, I001 -- Imports and configuration are shared across embedded cells.
    import hashlib
    import json
    import math
    import re
    import sys
    import uuid
    from contextlib import contextmanager
    from datetime import UTC, datetime
    from importlib import metadata
    from pathlib import Path as _EdaPath
    from subprocess import run as run_command
    from types import SimpleNamespace
    from unittest.mock import patch


    if bool(RAW_ARTIFACT_REF) == bool(RAW_MANIFEST_PATH):
        raise ValueError("Isi tepat satu input raw.")
    RUN_DIR = OUTPUT_DIR.expanduser().resolve() / f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}"
    RUN_DIR.mkdir(parents=True, exist_ok=False)

    PINS = {"ultralytics": "8.4.155", "wandb": "0.28.1"}
    PRESERVED_TORCH = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = [f"{package}=={version}" for package, version in PINS.items()
              if not any(dist.version == version for dist in metadata.distributions(name=package))]
    if needed:
        if not INSTALL_DEPENDENCIES:
            raise RuntimeError(f"Instal dependency baseline di kernel ini: {needed}")
        protected = RUN_DIR / "preserve_torch.txt"
        protected.write_text("".join(f"{name}=={version}\n" for name, version in PRESERVED_TORCH.items()), encoding="utf-8")
        run_command([sys.executable, "-m", "pip", "install", "--constraint", str(protected), *needed], check=True)
        if any(metadata.version(name) != version for name, version in PRESERVED_TORCH.items()):
            raise RuntimeError("Versi Torch/torchvision kernel berubah selama instalasi.")

    import matplotlib.pyplot as _eda_plt
    import pandas as pd
    import torch
    import ultralytics
    import wandb
    import yaml
    from thop import profile as thop_profile
    from ultralytics import YOLO
    from ultralytics.models.yolo.detect import DetectionTrainer
    from ultralytics.nn.modules import Detect
    from ultralytics.nn.modules.block import AAttn, Attention
    from ultralytics.nn.tasks import DetectionModel as _RecoveryDetectionModel
    from ultralytics.utils.torch_utils import _attention_ops

    _recovery_torch = torch
    _recovery_hashlib = hashlib
    _eda_math = math
    if (ultralytics.__version__, wandb.__version__) != tuple(PINS.values()):
        raise RuntimeError("Restart kernel setelah memasang versi dependency baseline.")
    if MODE not in {"full_val", "smoke"} or SEED not in {42, 0, 2024, 7, 3407}:
        raise ValueError("Pilih mode full_val/smoke dan satu seed yang tersedia pada baseline.")
    if DEVICE != "cpu" and not torch.cuda.is_available():
        raise RuntimeError("Pilih kernel GPU atau gunakan DEVICE='cpu' dalam smoke test lokal.")
    if MODE == "full_val" and DEVICE == "cpu":
        raise RuntimeError("Full validation protokol baseline memerlukan GPU.")

    NOTEBOOK_VARIANT = "s"
    RECOVERY_METHOD = "ft-baseline"
    RECOVERY_EPOCHS = 1 if MODE == "smoke" else EPOCHS
    POLTRADA_V3_ID = "ilhamrafiqin/poltrada-v3-yolo26-clean"
    POLTRADA_V3_MOUNT = Path("/kaggle/input/datasets") / POLTRADA_V3_ID
    DATASET_ID = globals().get("DATASET_ID", POLTRADA_V3_ID)
    BASELINE_REF = "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
    BASELINE_SHA256 = "5bc3ae61f8da1c9f03916922de10146805c27c64a11e46edd766a00d0f6f18fd"

    ENVIRONMENT = {"python": sys.version.split()[0], "kernel": sys.executable, "torch": torch.__version__,
                   "torchvision": metadata.version("torchvision"), "ultralytics-thop": metadata.version("ultralytics-thop"),
                   **PINS, "cuda": torch.version.cuda,
                   "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE != "cpu" else None,
                   "dataset_root": str(DATASET_ROOT), "output": str(RUN_DIR)}
    print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))


    def sha256_file(path):
        with Path(path).open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest()

In [ ]:
if RAW_ARTIFACT_REF or RAW_MANIFEST_PATH:
    def write_json(path, value):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
        temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
        os.replace(temporary, path)

    def write_csv(path, rows):
        import csv
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
        keys = list(dict.fromkeys((key for row in rows for key in row)))
        with temporary.open('w', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=keys)
            writer.writeheader()
            writer.writerows(({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value for key, value in row.items()} for row in rows))
        os.replace(temporary, path)

    def workflow_digest(value):
        import hashlib
        return hashlib.sha256(json.dumps(value, sort_keys=True, default=str, allow_nan=False).encode()).hexdigest()

    def workflow_bind(folder, name, contract):
        """Reject semantic changes on resume, allowing absolute paths to relocate."""
        path = Path(folder) / name
        contract = json.loads(json.dumps(contract, sort_keys=True, default=str, allow_nan=False))
        record = {'contract': contract, 'sha256': workflow_digest(contract)}
        if path.is_file() and json.loads(path.read_text(encoding='utf-8')) != record:
            raise ValueError(f'Kontrak resume berbeda: {name}. Gunakan suite baru.')
        write_json(path, record)
        return record

    def workflow_run_name(variant, target, recovery=False):
        tag = f'{target:.2f}'.replace('.', 'p')
        method = globals().get('RECOVERY_METHOD', 'finetune')
        recovery_tag = f'_ft{RECOVERY_EPOCHS}' if method == 'finetune' else f'_{method}{RECOVERY_EPOCHS}'
        ft = recovery_tag if recovery else ''
        smoke = '_smoke' if MODE == 'smoke' else ''
        return f'prun_lamp_gflops_yolo26{variant}_{PLATFORM}{ft}_su{tag}_seed{SEED}{smoke}'

    def workflow_login():
        import getpass
        import wandb
        key = os.environ.get('WANDB_API_KEY')
        if not key and PLATFORM == 'kaggle':
            try:
                from kaggle_secrets import UserSecretsClient
                key = UserSecretsClient().get_secret('WANDB_API_KEY')
            except (ImportError, RuntimeError, ValueError):
                pass
        if not key:
            key = getpass.getpass('WANDB_API_KEY: ')
        if not key:
            raise ValueError('W&B memerlukan API key untuk publikasi artefak.')
        os.environ['WANDB_API_KEY'] = key
        wandb.login(key=key, relogin=False)

    def workflow_version_ref(reference):
        if not isinstance(reference, str) or not re.fullmatch('[^/:]+/[^/:]+/[^/:]+:v\\d+', reference):
            raise ValueError('Input W&B memerlukan entity/project/artifact:vN (versi tetap).')
        return reference

    @contextmanager
    def workflow_tracking(folder, variant, target, *, recovery=False, source_ref=None, group=None, registration=False):
        import wandb
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        run = None
        if not WANDB_TRACKING:
            yield None
            return
        identity_path = folder / 'wandb_identity.json'
        existed = identity_path.is_file()
        name = workflow_run_name(variant, target, recovery) + ('_register_raw' if registration else '')
        identity = json.loads(identity_path.read_text(encoding='utf-8')) if existed else {'id': uuid.uuid4().hex, 'name': name, 'group': group or f'lamp-gflops-{PLATFORM}-{folder.parent.name}'}
        if identity['name'] != name:
            raise ValueError('Identitas run resume berbeda dari konfigurasi.')
        write_json(identity_path, identity)
        try:
            workflow_login()
            run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=name, id=identity['id'], resume='allow' if existed else 'never', group=identity['group'], job_type='register-raw' if registration else globals().get('RECOVERY_METHOD', 'finetune') + '-recovery' if recovery else 'pruning-no-recovery', tags=[PLATFORM, f'yolo26{variant}', 'lamp', 'depgraph', f'seed{SEED}', 'recovery' if recovery else 'raw-pruned', MODE], config={'target_speedup': target, 'model_variant': variant, 'platform': PLATFORM, 'seed': SEED, 'mode': MODE})
            if source_ref:
                run.use_artifact(workflow_version_ref(source_ref), type='model')
            yield run
        finally:
            if run is not None:
                run.finish()

    class WorkflowUploadPending(RuntimeError):
        """Local computation is committed; only publication needs retrying."""

    def workflow_publish(run, folder, label, files, *, metadata_info=None, aliases=None, event=None, kind='model'):
        import wandb
        folder = Path(folder)
        path = folder / 'artifact_publications.json'
        publications = json.loads(path.read_text(encoding='utf-8')) if path.is_file() else {}
        files = {name: Path(value).resolve(strict=True) for name, value in files.items()}
        hashes = {name: sha256_file(value) for name, value in files.items()}
        key = label + (':' + event if event else '')
        prior = publications.get(key)
        if prior and prior.get('status') == 'CONFIRMED' and (prior.get('files_sha256') == hashes):
            return prior
        record = {'status': 'NOT_RUN', 'files_sha256': hashes, 'label': label, 'event': event}
        if run is None:
            publications[key] = record
            write_json(path, publications)
            return record
        try:
            artifact = wandb.Artifact(f'{run.name}-{run.id}-{label}', type=kind, metadata={**(metadata_info or {}), 'files_sha256': hashes})
            for name, value in files.items():
                artifact.add_file(str(value), name=name)
            uploaded = run.log_artifact(artifact, aliases=aliases or ['latest'])
            uploaded.wait(timeout=300)
            reference = workflow_version_ref(uploaded.qualified_name)
            record.update(status='CONFIRMED', reference=reference)
        except Exception as error:
            record.update(status='FAIL', error=safe_error(error))
            publications[key] = record
            write_json(path, publications)
            raise WorkflowUploadPending(safe_error(error)) from error
        publications[key] = record
        write_json(path, publications)
        return record

    def workflow_dataset_manifest(dataset, source_yaml, root, folder):
        """Portable full train/val content identity, never an evaluation train=val placeholder."""
        raw = yaml.safe_load(Path(source_yaml).read_text(encoding='utf-8-sig'))
        names = ordered_names(raw['names'])
        root = Path(root).resolve()
        train = _gflops_split_images(root, raw.get('train'))
        val = _gflops_split_images(root, raw.get('val'))
        if any((image.with_suffix('.npy').is_file() for image in (*train, *val))):
            raise ValueError('Gunakan sumber dataset tanpa cache gambar .npy.')
        train_fp, train_rows = _gflops_fingerprint(root, train, 'train', names)
        val_fp, val_rows = _gflops_fingerprint(root, val, 'val', names)
        if {row['image_sha256'] for row in train_rows}.intersection((row['image_sha256'] for row in val_rows)):
            raise ValueError('Train dan val berbagi konten gambar; recovery tidak boleh memakai train=val.')
        record = {'format_version': 1, 'dataset_id': DATASET_ID, 'class_names': names, 'train_fingerprint': train_fp, 'val_fingerprint': val_fp, 'provenance_status': dataset.get('provenance_status', 'UNVERIFIED'), 'train_count': len(train), 'val_count': len(val), 'source_yaml_sha256': sha256_file(source_yaml)}
        write_json(Path(folder) / 'dataset_manifest.json', record)
        return (record, train, val)

    def _gflops_split_images(root, entries):
        """Resolve directories and image lists using dataset-relative identities."""
        from ultralytics.data.utils import IMG_FORMATS
        if isinstance(entries, str):
            entries = [entries]
        if not isinstance(entries, list) or not entries:
            raise ValueError('Split Poltrada harus berupa direktori atau daftar gambar lokal.')
        images = []
        for entry in entries:
            if not isinstance(entry, str) or not entry.strip():
                raise ValueError('Path split kosong atau tidak valid.')
            path = (root / entry).resolve()
            if (not path.exists() or not path.is_relative_to(root)) and entry.startswith('../'):
                path = (root / entry.removeprefix('../')).resolve()
            if not path.is_relative_to(root):
                raise ValueError(f'Path split berada di luar DATASET_ROOT: {path}')
            if path.is_dir():
                files = (item for item in path.rglob('*') if item.is_file())
            elif path.is_file() and path.suffix.lower() == '.txt':
                files = (path.parent / line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip())
            else:
                raise FileNotFoundError(f'Direktori atau daftar gambar split tidak tersedia: {path}')
            for image in files:
                if image.suffix.lower().removeprefix('.') not in IMG_FORMATS:
                    continue
                image = image.resolve()
                if not image.is_file() or not image.is_relative_to(root):
                    raise ValueError(f'Gambar tidak tersedia atau berada di luar DATASET_ROOT: {image}')
                images.append(image)
        if not images:
            raise ValueError('Split Poltrada tidak berisi gambar yang dapat dievaluasi.')
        if len(set(images)) != len(images):
            raise ValueError('Split Poltrada memuat path gambar duplikat.')
        return sorted(images, key=lambda image: image.relative_to(root).as_posix())

    def _gflops_fingerprint(root, images, split, names):
        """Hash every selected image and label; missing and empty labels stay distinct."""
        from ultralytics.data.utils import img2label_paths
        records = []
        for image, label in zip(images, img2label_paths([str(image) for image in images]), strict=True):
            label = Path(label).resolve()
            if not label.is_relative_to(root):
                raise ValueError(f'Label berada di luar DATASET_ROOT: {label}')
            records.append({'image': image.relative_to(root).as_posix(), 'image_sha256': sha256_file(image), 'label': label.relative_to(root).as_posix(), 'label_sha256': sha256_file(label) if label.is_file() else None})
        payload = json.dumps({'split': split, 'class_names': names, 'records': records}, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
        return (hashlib.sha256(payload).hexdigest(), records)

    def gflops_prepare_dataset(mode, data_yaml, dataset_root, suite_dir, *, seed, expected_fingerprint=None, dataset_id=None):
        """Freeze the input set before evaluating any model in the suite."""
        import yaml
        if mode not in {'smoke', 'full_val', 'test'}:
            raise ValueError('Mode evaluasi harus smoke, full_val, atau test.')
        if expected_fingerprint is not None:
            if not isinstance(expected_fingerprint, str) or len(expected_fingerprint) != 64 or any((char not in '0123456789abcdefABCDEF' for char in expected_fingerprint)):
                raise ValueError('Fingerprint full val harus berupa SHA-256 heksadesimal.')
            expected_fingerprint = expected_fingerprint.lower()
        source = Path(data_yaml).expanduser().resolve()
        if not source.is_file():
            return None
        root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.parent
        if not root.is_dir():
            raise FileNotFoundError(f'DATASET_ROOT tidak tersedia: {root}')
        raw = yaml.safe_load(source.read_text(encoding='utf-8-sig'))
        if not isinstance(raw, dict) or not isinstance(raw.get('names'), (dict, list)) or (not raw['names']):
            raise ValueError('DATA_YAML harus memuat names Poltrada yang berurutan.')
        class_names = ordered_names(raw['names'])
        if raw.get('nc', len(class_names)) != len(class_names):
            raise ValueError('nc di DATA_YAML berbeda dari jumlah names.')
        split = 'test' if mode == 'test' else 'val'
        val_images = _gflops_split_images(root, raw.get('val'))
        full_val_fingerprint = None
        if mode != 'smoke' or expected_fingerprint is not None:
            full_val_fingerprint, val_records = _gflops_fingerprint(root, val_images, 'val', class_names)
            if expected_fingerprint is not None and full_val_fingerprint != expected_fingerprint:
                raise ValueError('Fingerprint full val tidak cocok dengan konfigurasi yang dikunci.')
        official_mount = root == POLTRADA_V3_MOUNT.resolve() and source.is_relative_to(root)
        verified_copy = dataset_id == POLTRADA_V3_ID and expected_fingerprint is not None
        provenance_status = 'VERIFIED' if official_mount or verified_copy else 'UNVERIFIED'
        provenance_basis = 'KAGGLE_MOUNT' if official_mount else 'EXPECTED_FULL_VAL_FINGERPRINT' if verified_copy else 'USER_PATH_ONLY'
        if mode == 'test':
            if not raw.get('test') or not raw.get('train'):
                return None
            images = _gflops_split_images(root, raw['test'])
            fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
            test_hashes = {row['image_sha256'] for row in records}
            if any((row['image_sha256'] in test_hashes for row in val_records)):
                raise ValueError('Split test berbagi konten gambar dengan full val.')
            train_images = _gflops_split_images(root, raw['train'])
            if any((sha256_file(image) in test_hashes for image in train_images)):
                raise ValueError('Split test berbagi konten gambar dengan train.')
        elif mode == 'smoke':
            ranked = sorted(val_images, key=lambda image: (hashlib.sha256(f'{seed}:{image.relative_to(root).as_posix()}'.encode()).hexdigest(), image.relative_to(root).as_posix()))
            images = sorted(ranked[:128], key=lambda image: image.relative_to(root).as_posix())
            fingerprint, records = _gflops_fingerprint(root, images, split, class_names)
        else:
            images = val_images
            fingerprint, records = (full_val_fingerprint, val_records)
        if any((image.with_suffix('.npy').is_file() for image in images)):
            raise ValueError('Cache gambar .npy berada di samping citra evaluasi. Gunakan salinan dataset tanpa cache.')
        folder = Path(suite_dir).resolve()
        folder.mkdir(parents=True, exist_ok=True)
        paths = folder / f'evaluation_{mode}_images.txt'
        paths.write_text(''.join((f'{image.as_posix()}\n' for image in images)), encoding='utf-8')
        descriptor = {'path': str(root), 'train': str(paths), 'val': str(paths), 'names': class_names, 'nc': len(class_names)}
        if split == 'test':
            descriptor['test'] = str(paths)
        evaluation_yaml = folder / f'evaluation_{mode}_data.yaml'
        evaluation_yaml.write_text(yaml.safe_dump(descriptor, sort_keys=False, allow_unicode=True), encoding='utf-8')
        manifest = {'mode': mode, 'split': split, 'seed': seed, 'image_count': len(images), 'full_val_image_count': len(val_images), 'fingerprint': fingerprint, 'full_val_fingerprint': full_val_fingerprint, 'provenance_status': provenance_status, 'provenance_basis': provenance_basis, 'dataset_id': POLTRADA_V3_ID if official_mount else dataset_id, 'source_yaml': str(source), 'source_yaml_sha256': sha256_file(source), 'evaluation_yaml_sha256': sha256_file(evaluation_yaml), 'class_names': class_names, 'test_content_disjoint_from_train_val': True if mode == 'test' else None}
        manifest_path = folder / f'evaluation_{mode}_manifest.json'
        write_json(manifest_path, {**manifest, 'files': records})
        return {**manifest, 'manifest_path': str(manifest_path), 'yaml': str(evaluation_yaml), 'info': descriptor, 'val_paths': [str(paths)], 'image_files': [str(image) for image in images]}

    @contextmanager
    def gflops_disable_label_cache():
        """Force label parsing from the fingerprinted files without changing the input dataset."""
        from ultralytics.data import dataset as sdk_dataset

        def set_in_memory_version(_prefix, _path, records, version):
            records['version'] = version
        with patch.object(sdk_dataset, 'load_dataset_cache_file', side_effect=FileNotFoundError), patch.object(sdk_dataset, 'save_dataset_cache_file', side_effect=set_in_memory_version):
            yield

    def model_state_sha256(core):
        """Buktikan setiap konfigurasi dimulai dari bobot/buffer FP32 yang identik."""
        digest = hashlib.sha256()
        for name, value in sorted(core.state_dict().items()):
            tensor = value.detach().cpu().contiguous()
            digest.update(name.encode())
            digest.update(str((str(tensor.dtype), tuple(tensor.shape))).encode())
            digest.update(tensor.numpy().tobytes())
        return digest.hexdigest()

    def safe_error(error):
        message = str(error)
        secret = os.environ.get('WANDB_API_KEY')
        return message.replace(secret, '[REDACTED]') if secret else message

    def recovery_ordered_names(names):
        """Preserve class order and reject sparse or non-integer class identifiers."""
        if isinstance(names, dict):
            if set(names) != set(range(len(names))):
                raise ValueError('Recovery class IDs must be contiguous integers starting at zero.')
            names = [names[index] for index in range(len(names))]
        if not isinstance(names, (list, tuple)) or not names or (not all((isinstance(n, str) for n in names))):
            raise ValueError('Recovery requires an explicit nonempty ordered class-name list.')
        return list(names)

    def recovery_structure_signature(model):
        """Parameter shapes and physical Conv/BN dimensions; independent of learned values."""
        return {'parameters': {name: list(value.shape) for name, value in model.named_parameters()}, 'layers': {name: {'type': type(module).__name__, 'in': module.in_channels, 'out': module.out_channels, 'groups': module.groups} if isinstance(module, _recovery_torch.nn.Conv2d) else {'type': type(module).__name__, 'features': module.num_features} for name, module in model.named_modules() if isinstance(module, (_recovery_torch.nn.Conv2d, _recovery_torch.nn.BatchNorm2d))}}

    def recovery_state_sha256(model, *, parameters_only=False):
        digest = _recovery_hashlib.sha256()
        values = dict(model.named_parameters()) if parameters_only else model.state_dict()
        for name, value in sorted(values.items()):
            tensor = value.detach().cpu().contiguous()
            digest.update(name.encode())
            digest.update(str(tuple(tensor.shape)).encode())
            digest.update(str(tensor.dtype).encode())
            digest.update(tensor.numpy().tobytes())
        return digest.hexdigest()

    def _recovery_finite_state(model):
        return all((_recovery_torch.isfinite(v).all().item() for v in model.state_dict().values() if isinstance(v, _recovery_torch.Tensor) and v.is_floating_point()))

    ordered_names = recovery_ordered_names

    def recovery_read_json(path):
        return json.loads(Path(path).read_text(encoding='utf-8'))

    def recovery_raw_reference(source, files, folder, target):
        if not WANDB_TRACKING:
            return None
        if source['reference']:
            return workflow_version_ref(source['reference'])
        journal = source['path'].parent / 'artifact_publications.json'
        publication = recovery_read_json(journal).get('raw-pruned') if journal.is_file() else None
        hashes = {name: sha256_file(value) for name, value in files.items()}
        if publication and publication.get('status') == 'CONFIRMED' and (publication.get('files_sha256') == hashes):
            return workflow_version_ref(publication['reference'])
        registry = folder / 'register_raw'
        workflow_bind(registry, 'input_contract.json', {'files_sha256': hashes, 'platform': PLATFORM, 'variant': NOTEBOOK_VARIANT, 'target': target, 'mode': MODE, 'seed': SEED})
        with workflow_tracking(registry, NOTEBOOK_VARIANT, target, registration=True) as run:
            record = workflow_publish(run, registry, 'raw-pruned', files, aliases=['raw-pruned', 'latest'], metadata_info={'stage': 'raw_pruned', 'raw_sha256': hashes['pruned_fp32.pt']})
        return workflow_version_ref(record['reference'])

## 03 — Trainer native, full validation dan EDA

In [ ]:
if RAW_ARTIFACT_REF or RAW_MANIFEST_PATH:
    def _validation_number(value):
        """JSON scalar: unsupported/nonfinite auxiliary metrics are unavailable, not zero."""
        if value is None:
            return None
        try:
            number = float(value)
        except (TypeError, ValueError, OverflowError):
            return None
        return number if math.isfinite(number) else None

    def _validation_array(value):
        import numpy as np
        if hasattr(value, 'detach'):
            value = value.detach().cpu().numpy()
        return np.asarray(value, dtype=float)

    def _validation_per_class(metric, class_names):
        """Map SDK result rows via ap_class_index, never by names or array position."""
        import numpy as np
        box = metric.box
        support = getattr(metric, 'nt_per_class', None)
        if support is not None:
            support = _validation_array(support)
            if support.shape != (len(class_names),) or not np.isfinite(support).all() or (support < 0).any() or (not np.equal(support, np.floor(support)).all()):
                raise ValueError('Support kelas SDK tidak valid atau tidak cocok dengan jumlah kelas.')
        rows = [{'class_id': i, 'name': name, 'ap50': None, 'ap50_95': None, 'precision': None, 'recall': None, 'f1': None, 'support': int(support[i]) if support is not None else None} for i, name in enumerate(class_names)]
        indices = list(getattr(box, 'ap_class_index', []))
        validated_indices = []
        for value in indices:
            scalar = _validation_number(value)
            if scalar is None or scalar != int(scalar) or (not 0 <= int(scalar) < len(rows)):
                raise ValueError('Indeks kelas SDK tidak valid.')
            validated_indices.append(int(scalar))
        if len(set(validated_indices)) != len(validated_indices):
            raise ValueError('Indeks kelas SDK duplikat.')
        for offset, class_id in enumerate(validated_indices):
            for key, attribute in (('ap50', 'ap50'), ('ap50_95', 'ap'), ('precision', 'p'), ('recall', 'r'), ('f1', 'f1')):
                values = getattr(box, attribute, None)
                if values is not None and len(values) > offset:
                    rows[class_id][key] = _validation_number(values[offset])
        return (rows, validated_indices)

    def _eda_finite(value):
        return isinstance(value, (int, float)) and (not isinstance(value, bool)) and _eda_math.isfinite(value)

    def _eda_save_figure(fig, path):
        path = _EdaPath(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path.with_suffix('.png'), dpi=int(globals().get('EDA_DPI', 300)), bbox_inches='tight')
        fig.savefig(path.with_suffix('.svg'), bbox_inches='tight')
        _eda_plt.close(fig)
        return {'png': path.with_suffix('.png').name, 'svg': path.with_suffix('.svg').name}

    """EDA for measured baseline, raw pruned, and recovered DetectionModels."""
    # ruff: noqa: F821 -- Notebook cells supply shared persistence and EDA declarations.

    RECOVERY_STAGE_STYLES = {
        "baseline": ("Baseline", "#2863AD"),
        "raw_pruned": ("Raw pruned", "#BA492E"),
        "recovered": ("Recovered", "#217854"),
    }
    RECOVERY_ACCURACY_FIELDS = {"map50_95": "mAP50–95", "map50": "mAP50", "precision": "Precision", "recall": "Recall"}
    RECOVERY_CLASS_FIELDS = {"ap50_95": "AP50–95", "ap50": "AP50", "precision": "Precision", "recall": "Recall", "f1": "F1"}
    RECOVERY_LATENCY_CONTRACT = ("device", "precision", "batch", "imgsz", "fusion", "scope", "warmup", "repeats")
    RECOVERY_EDA_METHODS = {"finetune": "Fine-tuning standar", "native": "Native KD", "crosskd": "CrossKD", "mgd": "MGD"}


    def _recovery_eda_num(value):
        return value if _eda_finite(value) else None


    def _recovery_eda_ratio(before, after):
        return before / after if before is not None and after is not None and after > 0 else None


    def _recovery_eda_delta(after, before):
        return 100 * (after - before) if after is not None and before is not None else None


    def _recovery_eda_metadata(result):
        return {key: result.get(key) for key in ("model_variant", "method", "platform", "mode", "configuration_id",
                                                 "target_speedup", "target_status", "alias_of", "raw_sha256")}


    def recovery_comparison_rows(result):
        """Use successful measurements only and join class data by stable ID."""
        stages = result.get("stages", {})
        metadata = _recovery_eda_metadata(result)
        rows, per_class_by_stage, class_names = [], {}, {}
        latency_rows = [stage.get("latency", {}) for stage in stages.values()]
        latency_contracts = [tuple(item.get(key) for key in RECOVERY_LATENCY_CONTRACT)
                             for item in latency_rows if item.get("status") in {"PASS", "MEASURED"}]
        comparable_latency = all(contract == latency_contracts[0] for contract in latency_contracts)
        for stage in RECOVERY_STAGE_STYLES:
            measured = stages.get(stage, {})
            validation_status = measured.get("validation_status", "NOT_RUN")
            validation = measured.get("validation", {}) if validation_status == "PASS" else {}
            latency = measured.get("latency", {})
            latency_status = latency.get("status", "NOT_RUN")
            if latency_status in {"PASS", "MEASURED"} and not comparable_latency:
                latency_status = "NOT_COMPARABLE"
            latency = latency if latency_status in {"PASS", "MEASURED"} else {}
            rows.append({**metadata, "stage": stage, "validation_status": validation_status, "latency_status": latency_status,
                         "validation_image_count": validation.get("validation_image_count"),
                         "validation_fingerprint": validation.get("dataset_fingerprint"),
                         "validation_device": validation.get("validation_arguments_effective", {}).get("device"),
                         "latency_device": latency.get("device"), "latency_scope": latency.get("scope"),
                         **{key: _recovery_eda_num(validation.get(key)) for key in RECOVERY_ACCURACY_FIELDS},
                         **{key: _recovery_eda_num(measured.get(key)) for key in ("parameters", "gflops", "checkpoint_bytes", "conv_output_channels")},
                         "latency_median_ms": _recovery_eda_num(latency.get("median_ms")),
                         "latency_p95_ms": _recovery_eda_num(latency.get("p95_ms"))})
            indexed = {}
            for item in validation.get("per_class", []):
                class_id, name = item["class_id"], item["name"]
                if class_id in indexed or class_names.setdefault(class_id, name) != name:
                    raise ValueError(f"Identitas kelas tidak konsisten: {stage}, {class_id}, {name}")
                indexed[class_id] = item
            per_class_by_stage[stage] = indexed
        baseline, raw, recovered = rows
        class_rows = []
        for row in rows:
            stage = row["stage"]
            row["gflops_speedup"] = _recovery_eda_ratio(baseline["gflops"], row["gflops"])
            row["latency_speedup"] = _recovery_eda_ratio(baseline["latency_median_ms"], row["latency_median_ms"])
            for key in RECOVERY_ACCURACY_FIELDS:
                row[key + "_drop_pp"] = _recovery_eda_delta(baseline[key], row[key])
                row[key + "_recovery_gain_pp"] = _recovery_eda_delta(recovered[key], raw[key]) if stage == "recovered" else None
            for key, name in (("parameters", "parameter"), ("gflops", "gflops"), ("conv_output_channels", "channel"),
                              ("checkpoint_bytes", "checkpoint")):
                ratio = _recovery_eda_ratio(row[key], baseline[key])
                row[name + "_reduction_pct"] = None if ratio is None else 100 * (1 - ratio)
            for class_id, name in sorted(class_names.items()):
                record = per_class_by_stage[stage].get(class_id, {})
                class_row = {**metadata, "stage": stage, "class_id": class_id, "name": name,
                             "support": _recovery_eda_num(record.get("support")),
                             **{key: _recovery_eda_num(record.get(key)) for key in RECOVERY_CLASS_FIELDS}}
                for key in ("ap50_95", "ap50"):
                    class_row[key + "_drop_pp"] = _recovery_eda_delta(
                        _recovery_eda_num(per_class_by_stage["baseline"].get(class_id, {}).get(key)), class_row[key])
                    class_row[key + "_recovery_gain_pp"] = _recovery_eda_delta(
                        class_row[key], _recovery_eda_num(per_class_by_stage["raw_pruned"].get(class_id, {}).get(key))
                    ) if stage == "recovered" else None
                class_rows.append(class_row)
        return rows, class_rows


    def recovery_epoch_rows(rows):
        """Flatten epoch records for plotting and logging without inventing missing loss values."""
        return [{"epoch": row["epoch"], "optimizer_steps": row.get("optimizer_steps"), "fitness": row.get("fitness"),
                 **{f"train/{name.removeprefix('train/')}": _recovery_eda_num(value)
                    for name, value in row.get("losses", {}).items()},
                 **{name: _recovery_eda_num(value) for name, value in row.get("metrics", {}).items()},
                 **{f"lr/pg{index}": _recovery_eda_num(value) for index, value in enumerate(row.get("learning_rates", []))}}
                for row in rows]


    def _recovery_draw_bar_panel(ax, records, *, title, unit, bounded=False):
        groups = list(dict.fromkeys(record["group"] for record in records))
        labels = {record["group"]: record["group_label"] for record in records}
        positions = {group: index for index, group in enumerate(groups)}
        width = .24
        plotted = 0
        for offset, (stage, (_label, color)) in enumerate(RECOVERY_STAGE_STYLES.items()):
            observed = [record for record in records if record["stage"] == stage and _eda_finite(record["value"])]
            if observed:
                bars = ax.bar([positions[record["group"]] + (offset - 1) * width for record in observed],
                              [record["value"] for record in observed], width=width, color=color)
                if len(groups) <= 4:
                    ax.bar_label(bars, labels=[f"{record['value']:.4g}" for record in observed], fontsize=7, padding=2)
                plotted += len(observed)
        if not plotted:
            ax.text(.5, .55, "Belum terukur", ha="center", va="center", transform=ax.transAxes, color="#57616B")
        ax.set_xticks(list(positions.values()), [labels[group] for group in groups],
                      rotation=30 if len(groups) > 4 else 0, ha="right" if len(groups) > 4 else "center")
        ax.set_title(title)
        ax.set_ylabel(unit)
        if bounded:
            ax.set_ylim(0, 1.1)
        else:
            ax.margins(y=.15)
        ax.set_axisbelow(True)
        ax.grid(axis="y", alpha=.18)
        return plotted




    def _recovery_epoch_plots(rows, folder):
        losses = sorted({key.split('/', 1)[1] for row in rows for key in row if key.startswith(('train/', 'val/'))})
        groups = [(key, (f'train/{key}', f'val/{key}'), 'Loss') for key in losses]
        groups.extend((('Akurasi validation', ('metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)'), 'Proporsi'), ('Learning rate', sorted({key for row in rows for key in row if key.startswith('lr/')}), 'Learning rate'), ('Optimizer', ('optimizer_steps',), 'Jumlah langkah kumulatif')))
        files = []
        for index, (title, keys, ylabel) in enumerate(groups):
            observed = {key: [(row['epoch'], row[key]) for row in rows if _eda_finite(row.get(key))] for key in keys}
            if not any(observed.values()):
                continue
            fig, ax = _eda_plt.subplots(figsize=(9, 4))
            for key, points in observed.items():
                if points:
                    ax.plot(*zip(*points), label=key, marker='.' if len(points) < 10 else None)
            ax.set(title=title, xlabel='Epoch', ylabel=ylabel)
            ax.grid(alpha=0.18)
            ax.legend(frameon=False)
            fig.tight_layout()
            files.append(_eda_save_figure(fig, folder / f'epochs_{index:02d}'))
        return files

    def recovery_log_comparison(run, result, folder):
        """Log the exact local tables and comparison figures to the recovery run."""
        import wandb
        rows, classes = recovery_comparison_rows(result)
        payload = {}
        for name, records in (('three_stage_metrics', rows), ('three_stage_per_class', classes)):
            if records:
                keys = list(dict.fromkeys((key for row in records for key in row)))
                payload[f'recovery/{name}'] = wandb.Table(columns=keys, data=[[row.get(key) for key in keys] for row in records])
        for path in sorted((_EdaPath(folder) / 'eda').glob('comparison_*.png')):
            payload[f'recovery/{path.stem}'] = wandb.Image(str(path))
        run.log(payload)

    def model_measurements(core, imgsz):
        """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
        parameters = sum((p.numel() for p in core.parameters()))
        copy = deepcopy(core).eval()
        parameter = next(copy.parameters())
        sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
        attention_macs = []

        def count_attention(module, inputs, output):
            counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim, key_dim=getattr(module, 'key_dim', module.head_dim), area=getattr(module, 'area', 1))
            _attention_ops(counter, inputs, output)
            attention_macs.append(float(counter.total_ops))
        handles = [module.register_forward_hook(count_attention) for module in copy.modules() if isinstance(module, (Attention, AAttn))]
        try:
            with torch.no_grad():
                macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
        finally:
            for handle in handles:
                handle.remove()
        gflops = (float(macs) + sum(attention_macs)) * 2 / 1000000000.0
        if not math.isfinite(gflops) or gflops <= 0:
            raise RuntimeError('Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.')
        return {'parameters': parameters, 'gflops': gflops, 'precision': 'FP32', 'fusion': False}

    """Standalone baseline-matched fine-tuning of a single pruned YOLO26s student."""
    # ruff: noqa: F821 -- Common declarations are embedded in the preceding notebook cells.
    # ruff: noqa: BLE001 -- Record failures per stage and retain local results.
    from copy import deepcopy


    def fine_tuned_source(folder):
        reference = None
        if RAW_ARTIFACT_REF:
            reference = workflow_version_ref(RAW_ARTIFACT_REF)
            workflow_login()
            artifact = wandb.Api().artifact(reference, type="model")
            path = Path(artifact.download(root=str(folder / "raw_artifact"))) / "raw_pruned_manifest.json"
        else:
            path = Path(RAW_MANIFEST_PATH).expanduser().resolve(strict=True)
        manifest = recovery_read_json(path)
        target = manifest.get("target_speedup")
        if (manifest.get("stage") != "raw_pruned" or manifest.get("format_version") != 2
                or manifest.get("model_variant") != "s" or type(target) not in {int, float}
                or not 1 < target <= 1.70 or manifest.get("configuration_id") != f"target-{target:.2f}x"
                or manifest.get("target_status") not in {"REACHED", "TARGET_NOT_REACHED"}
                or any(manifest.get(key) != "PASS" for key in ("structural_status", "oracle_status", "reload_status"))):
            raise ValueError("Input harus manifest raw pruned YOLO26s notebook 06 dengan audit struktur, oracle, dan reload PASS.")
        baseline = manifest["source_baseline"]
        if baseline["sha256"] != BASELINE_SHA256 or baseline["requested_artifact"] != BASELINE_REF:
            raise ValueError("Baseline sumber bukan YOLO26s v62 yang dikunci untuk pruning.")

        raw = manifest["raw_checkpoint"]
        saved_baseline = manifest["baseline_checkpoint"]
        raw_path = path.parent / raw["relative_path"]
        baseline_path = path.parent / saved_baseline["artifact_path"]
        if not baseline_path.is_file() and not reference:
            baseline_path = path.parent / saved_baseline["relative_path"].replace("\\", "/")
        files = {"raw_pruned_manifest.json": path, "pruned_fp32.pt": raw_path.resolve(strict=True),
                 "baseline_fp32.pt": baseline_path.resolve(strict=True)}
        if EXPECTED_RAW_SHA256 is not None and raw["sha256"] != EXPECTED_RAW_SHA256.lower():
            raise ValueError("SHA-256 raw berbeda dari student lock.")
        expected_files = {"pruned_fp32.pt": raw["sha256"], "baseline_fp32.pt": saved_baseline["sha256"]}
        for name, expected in manifest.get("audit_files", {}).items():
            if Path(name).name != name or name in files:
                raise ValueError(f"Nama file audit raw tidak valid: {name}")
            files[name] = path.parent / name
            expected_files[name] = expected
        for name, expected in expected_files.items():
            if sha256_file(files[name]) != expected:
                raise ValueError(f"SHA-256 input tidak cocok: {name}")
        return {"path": path, "manifest": manifest, "reference": reference}, files


    def fine_tuned_dataset(manifest, folder):
        identity = manifest["dataset"]
        if MODE == "full_val" and (manifest["evaluation_mode"] != "full_val" or manifest["validation_status"] != "PASS"
                                   or identity["provenance_status"] != "VERIFIED" or identity["dataset_id"] != POLTRADA_V3_ID):
            raise ValueError("Run penelitian membutuhkan raw hasil full validation Poltrada V3 VERIFIED.")
        expected_val = identity["fingerprint"] if manifest["evaluation_mode"] == "full_val" else None
        dataset = gflops_prepare_dataset(MODE, DATA_YAML, DATASET_ROOT, folder, seed=manifest["seed"],
                                        expected_fingerprint=expected_val, dataset_id=DATASET_ID)
        if dataset is None:
            raise FileNotFoundError("Atur DATASET_ROOT dan DATA_YAML Poltrada V3 dengan train dan val asli.")
        record, train, _ = workflow_dataset_manifest(dataset, DATA_YAML, DATASET_ROOT, folder)
        if (record["dataset_id"] != identity["dataset_id"] or record["class_names"] != identity["class_names"]
                or record["train_fingerprint"] != identity["train_fingerprint"]
                or (expected_val is None and dataset["fingerprint"] != identity["fingerprint"])):
            raise ValueError("Fingerprint data train/val atau urutan kelas berbeda dari manifest raw.")
        train_list = folder / "train_images.txt"
        train_list.write_text("".join(f"{image.as_posix()}\n" for image in train), encoding="utf-8")
        data_yaml = folder / "train_val_data.yaml"
        data_yaml.write_text(yaml.safe_dump({**dataset["info"], "train": str(train_list.resolve())},
                                           allow_unicode=True, sort_keys=False), encoding="utf-8")
        return dataset, record, train, data_yaml


    def fine_tuned_check_heads(core, names):
        if type(core) is not _RecoveryDetectionModel:
            raise ValueError("Diperlukan DetectionModel YOLO26 native dual-head.")
        head = core.model[-1]
        # Native validation sets the EMA inference flag to one-to-many while preserving both heads.
        if (type(head) is not Detect or recovery_ordered_names(core.names) != names or head.nc != len(names)
                or getattr(head, "one2one_cv2", None) is None or getattr(head, "one2one_cv3", None) is None
                or any(not isinstance(branch, torch.nn.ModuleList) or len(branch) != head.nl
                       for branches in (head.one2many, head.one2one) for branch in branches.values())):
            raise ValueError("Cabang Detect one-to-many/one-to-one atau urutan kelas tidak valid.")


    def fine_tuned_load_raw(path, manifest):
        # Only deserialize the trusted input package, after all file hashes match.
        payload = torch.load(path, map_location="cpu", weights_only=False)
        core = payload.get("model")
        fine_tuned_check_heads(core, manifest["dataset"]["class_names"])
        signature = recovery_structure_signature(core)
        serialized = {"parameters": signature["parameters"], "conv_bn": {
            name: [layer["in"], layer["out"], layer["groups"]] if "in" in layer else [layer["features"]]
            for name, layer in signature["layers"].items()}}
        if (payload.get("recovery_enabled") is not False or not core.end2end or serialized != manifest["structure_signature"]
                or model_state_sha256(core) != manifest["state_sha256"] or not _recovery_finite_state(core)
                or any(p.dtype != torch.float32 for p in core.parameters())):
            raise ValueError("Dimensi atau tensor checkpoint raw tidak cocok dengan manifest FP32.")
        return core


    class BaselinePrunedTrainer(DetectionTrainer):
        """Inject a verified pruned model and leave the native training loop intact."""

        def __init__(self, raw, *, overrides):
            self.raw = raw
            super().__init__(overrides=overrides)

        def setup_model(self):
            if self.resume or self.args.distill_model is not None:
                raise ValueError("Fine-tuning harus mulai dari student raw dengan optimizer baru dan loss native.")
            self.model = deepcopy(self.raw)
            self.model.criterion = None
            self.model.requires_grad_(True)

        def get_model(self, cfg=None, weights=None, verbose=True):
            raise RuntimeError("Rekonstruksi model dari YAML dilarang untuk student pruned.")

        def run_callbacks(self, event):
            for callback in self.callbacks.get(event, []):
                module = getattr(callback, "__module__", None) or ""
                if not module.startswith("ultralytics.utils.callbacks.") or module.endswith(".base"):
                    callback(self)


    def fine_tuned_check_images(loader, expected):
        observed = [Path(path).resolve() for path in loader.dataset.im_files]
        if len(observed) != len(expected) or set(observed) != {Path(path).resolve() for path in expected}:
            raise ValueError("Citra aktual dataloader berbeda dari split yang diverifikasi.")


    def fine_tuned_fit(raw, raw_path, data_yaml, train, dataset, folder, run, errors):
        options = fine_tuned_train_args(str(data_yaml), str(folder), "training")
        trainer = BaselinePrunedTrainer(raw, overrides={**options, "model": str(raw_path), "task": "detect", "mode": "train"})
        initial = recovery_state_sha256(raw)
        initial_parameters = recovery_state_sha256(raw, parameters_only=True)
        signature = recovery_structure_signature(raw)
        epochs = []

        def check_protocol(trainer):
            if trainer.batch_size != options["batch"] or trainer.amp != options["amp"]:
                raise RuntimeError("Batch atau AMP efektif berubah dari protokol yang diminta.")

        def check_start(trainer):
            check_protocol(trainer)
            fine_tuned_check_heads(trainer.model, dataset["class_names"])
            if (recovery_state_sha256(trainer.model) != initial
                    or recovery_structure_signature(trainer.model) != signature):
                raise ValueError("Bobot awal training harus identik dengan raw pruned.")
            fine_tuned_check_images(trainer.train_loader, train)
            fine_tuned_check_images(trainer.test_loader, dataset["image_files"])
            write_json(folder / "training_config.json", {
                "requested": options, "effective": vars(trainer.args), "amp_effective": trainer.amp,
                "training_batch": trainer.train_loader.batch_size,
                "training_validation_batch": trainer.test_loader.batch_size, "initial_state_sha256": initial,
                "optimizer_class": type(trainer.optimizer).__name__, "scheduler_class": type(trainer.scheduler).__name__,
            })

        def log_epoch(trainer):
            check_protocol(trainer)
            epoch = int(trainer.epoch) + 1
            row = {"epoch": epoch, "fitness": _validation_number(trainer.fitness),
                   "losses": {name: _validation_number(value) for name, value in trainer.tloss.items()},
                   "metrics": {name: _validation_number(value) for name, value in trainer.metrics.items()},
                   "learning_rates": [float(group["lr"]) for group in trainer.optimizer.param_groups]}
            epochs.append(row)
            write_json(folder / "epoch_metrics.json", epochs)
            if run is not None:
                try:
                    run.log(recovery_epoch_rows([row])[0], step=epoch)
                except Exception as error:
                    errors.append({"stage": "upload", "epoch": epoch, "error": safe_error(error)})

        trainer.add_callback("on_train_start", check_start)
        trainer.add_callback("on_train_epoch_start", check_protocol)
        trainer.add_callback("on_model_save", log_epoch)
        with gflops_disable_label_cache():
            trainer.train()
        if len(epochs) != options["epochs"] or not trainer.best.is_file():
            raise RuntimeError("Training belum menyelesaikan budget epoch atau best.pt tidak tersedia.")
        if recovery_state_sha256(trainer.model, parameters_only=True) == initial_parameters:
            raise RuntimeError("Bobot student tidak berubah selama training.")
        best = YOLO(str(trainer.best)).model
        fine_tuned_check_heads(best, dataset["class_names"])
        if recovery_structure_signature(best) != signature or not _recovery_finite_state(best):
            raise RuntimeError("Reload best.pt gagal mempertahankan struktur raw atau mengandung tensor nonfinite.")
        return trainer.best, epochs, {"initial_state_sha256": initial, "structure_signature": signature,
                                      "initial_parameters_sha256": initial_parameters,
                                      "trained_parameters_sha256": recovery_state_sha256(trainer.model, parameters_only=True)}


    def fine_tuned_measure(path, data_yaml, dataset, folder, stage):
        model = YOLO(str(path))
        fine_tuned_check_heads(model.model, dataset["class_names"])
        core = model.model
        core.end2end = True  # Match notebook 06 for profiling. Native val selects its own head mode.
        complexity = model_measurements(core, IMGSZ)
        channels = sum(module.out_channels for module in core.modules() if isinstance(module, torch.nn.Conv2d))
        effective = {}

        def capture(validator):
            fine_tuned_check_images(validator.dataloader, dataset["image_files"])
            fields = ("split", "imgsz", "batch", "conf", "iou", "max_det", "rect", "quantize", "augment", "workers", "nms")
            effective.update({key: getattr(validator.args, key, None) for key in fields}, device=str(validator.device),
                             end2end=bool(validator.end2end), dataloader_batch=validator.dataloader.batch_size)

        model.add_callback("on_val_end", capture)
        arguments = fine_tuned_val_args(str(data_yaml))
        with gflops_disable_label_cache():
            metrics = model.val(**arguments, project=str(folder), name=stage)
        overall = {name: _validation_number(getattr(metrics.box, attr)) for name, attr in
                   {"map50_95": "map", "map50": "map50", "precision": "mp", "recall": "mr"}.items()}
        if not effective or any(value is None for value in overall.values()):
            raise RuntimeError("Metrik validator baseline belum lengkap.")
        validation = {**overall, "validation_arguments_requested": arguments, "validation_arguments_effective": effective,
                      "validation_image_count": len(dataset["image_files"]), "dataset_fingerprint": dataset["fingerprint"],
                      "per_class": _validation_per_class(metrics, dataset["class_names"])[0],
                      "inference_ms_per_image": _validation_number(metrics.speed.get("inference"))}
        return {"validation_status": "PASS", "validation": validation, **complexity,
                "gflops_protocol": "notebook-06 / FP32 unfused dual-head", "conv_output_channels": channels,
                "checkpoint_bytes": Path(path).stat().st_size}


    def fine_tuned_comparison(result):
        rows, classes = recovery_comparison_rows(result)
        for row in rows:
            stage = result["stages"].get(row["stage"], {})
            row["inference_ms_per_image"] = stage.get("validation", {}).get("inference_ms_per_image")
        return rows, classes


    def fine_tuned_eda(result, folder, epochs):
        from matplotlib.patches import Patch

        folder = folder / "eda"
        rows, classes = fine_tuned_comparison(result)
        panels = [
            ("accuracy", "Metrik deteksi", "Proporsi", tuple(RECOVERY_ACCURACY_FIELDS.items()), True),
            ("drop", "Penurunan terhadap baseline", "Poin persentase", (("map50_95_drop_pp", "mAP50–95"),), False),
            ("gain", "Gain terhadap raw pruned", "Poin persentase", (("map50_95_recovery_gain_pp", "mAP50–95"),), False),
            ("parameters", "Parameter", "Jumlah parameter", (("parameters", "Parameter"),), False),
            ("gflops", "Komputasi", "GFLOPs", (("gflops", "Komputasi"),), False),
            ("inference", "Waktu inference dari validator", "Milidetik per citra",
             (("inference_ms_per_image", "Inference"),), False),
        ]
        plots = [(name, title, unit, bounded,
                  [{"group": key, "metric": key, "group_label": label, "stage": row["stage"], "value": row[key], "unit": unit}
                   for key, label in fields for row in rows]) for name, title, unit, fields, bounded in panels]
        for metric, label in (("ap50_95", "AP50–95"), ("ap50", "AP50")):
            plots.append((metric, f"{label} per kelas", "Proporsi", True,
                          [{"group": row["class_id"], "metric": metric, "group_label": row["name"], "stage": row["stage"],
                            "value": row[metric], "unit": "Proporsi"} for row in classes]))
        plot_data = []
        for name, title, unit, bounded, values in plots:
            plot_data.extend({"figure": name, **value} for value in values)
            fig, ax = _eda_plt.subplots(figsize=(8, 4.5))
            _recovery_draw_bar_panel(ax, values, title=title, unit=unit, bounded=bounded)
            ax.legend(handles=[Patch(color=color, label=label) for label, color in RECOVERY_STAGE_STYLES.values()])
            fig.tight_layout()
            _eda_save_figure(fig, folder / f"comparison_{name}")
        _recovery_epoch_plots(recovery_epoch_rows(epochs), folder / "training")
        for name, values in (("stage_metrics", rows), ("per_class_metrics", classes), ("plot_data", plot_data)):
            write_json(folder / f"{name}.json", values)
            write_csv(folder / f"{name}.csv", values)


    def fine_tuned_execute(folder):
        result = {"status": "NOT_RUN", "output": str(folder), "platform": PLATFORM, "model_variant": "s",
                  "method": "ft-baseline", "mode": MODE, "seed": SEED, "stages": {}, "errors": []}
        result.update({f"{stage}_status": "NOT_RUN" for stage in ("input", "training", "reload", "validation", "eda", "upload")})
        manifest_path = folder / "fine_tuned_manifest.json"
        stage = "input"
        try:
            source, files = fine_tuned_source(folder)
            manifest = source["manifest"]
            target = manifest["target_speedup"]
            dataset, record, train, data_yaml = fine_tuned_dataset(manifest, folder / "data")
            raw_path = files["pruned_fp32.pt"]
            raw = fine_tuned_load_raw(raw_path, manifest)
            raw_sha = manifest["raw_checkpoint"]["sha256"]
            result.update(raw_sha256=raw_sha, raw_pruning_seed=manifest["seed"], target_speedup=target,
                          target_status=manifest["target_status"], configuration_id=manifest["configuration_id"],
                          raw_gflops_speedup=manifest["actual_gflops_speedup"], dataset=record,
                          baseline_artifact_ref=BASELINE_REF, baseline_sha256=manifest["baseline_checkpoint"]["sha256"],
                          baseline_source_sha256=BASELINE_SHA256, environment=ENVIRONMENT, input_status="PASS")
            stage = "upload"
            raw_ref = source["reference"] or recovery_raw_reference(source, files, folder, target)
            result["raw_artifact_ref"] = raw_ref
            write_json(manifest_path, result)
            with workflow_tracking(folder, "s", target, recovery=True, source_ref=raw_ref) as run:
                if run is not None:
                    run.config.update({"protocol": "baseline-02", "raw_artifact_ref": raw_ref, "raw_sha256": raw_sha,
                                       "raw_pruning_seed": manifest["seed"], "target_status": manifest["target_status"],
                                       "actual_gflops_speedup": manifest["actual_gflops_speedup"], "dataset": record,
                                       "training": fine_tuned_train_args(str(data_yaml), str(folder), "training")})
                stage = "training"
                best, epochs, audit = fine_tuned_fit(raw, raw_path, data_yaml, train, dataset, folder, run, result["errors"])
                result.update(training_status="PASS", reload_status="PASS", epochs=len(epochs), **audit)
                result["checkpoint"] = {"relative_path": best.relative_to(folder).as_posix(), "sha256": sha256_file(best)}
                result["raw_unchanged"] = sha256_file(raw_path) == raw_sha
                if not result["raw_unchanged"]:
                    raise RuntimeError("SHA-256 raw berubah sesudah training.")
                model_manifest = folder / "fine_tuned_model.json"
                model_fields = ("model_variant", "configuration_id", "target_status", "raw_sha256", "raw_artifact_ref",
                                "seed", "structure_signature", "checkpoint", "epochs")
                write_json(model_manifest, {key: result[key] for key in model_fields})
                stage = "validation"
                for name, checkpoint in (("baseline", files["baseline_fp32.pt"]), ("raw_pruned", raw_path), ("recovered", best)):
                    result["stages"][name] = fine_tuned_measure(checkpoint, data_yaml, dataset, folder / "validation", name)
                    write_json(manifest_path, result)
                baseline, pruned, recovered = result["stages"].values()
                if any(row["validation"]["validation_arguments_effective"] != baseline["validation"]["validation_arguments_effective"]
                       for row in (pruned, recovered)) or any(pruned[key] != recovered[key] for key in ("parameters", "gflops")):
                    raise RuntimeError("Protokol evaluasi atau kompleksitas model raw/recovered berbeda.")
                result["validation_status"] = "PASS"
                stage = "eda"
                fine_tuned_eda(result, folder, epochs)
                result["eda_status"] = "PASS"
                if run is not None:
                    stage = "upload"
                    result["checkpoint_artifact"] = workflow_publish(run, folder, "fine-tuned", {
                        result["checkpoint"]["relative_path"]: best, model_manifest.name: model_manifest}, aliases=["best", "latest"],
                        metadata_info={"raw_sha256": raw_sha, "raw_artifact_ref": raw_ref})
                    recovery_log_comparison(run, result, folder)
                    reports = {path.relative_to(folder).as_posix(): path for sub in ("eda", "validation")
                               for path in (folder / sub).rglob("*") if path.suffix in {".png", ".svg", ".csv", ".json"}}
                    reports["training_config.json"] = folder / "training_config.json"
                    reports["epoch_metrics.json"] = folder / "epoch_metrics.json"
                    result["results_artifact"] = workflow_publish(run, folder, "fine-tuned-results", reports, kind="results")
                    result["upload_status"] = "PARTIAL" if result["errors"] else "PASS"
        except (Exception, KeyboardInterrupt) as error:
            result[f"{stage}_status"] = "FAIL"
            result["reason"] = safe_error(error) or "Interrupted"
            result["errors"].append({"stage": stage, "error": result["reason"]})
        required = ["input", "training", "reload", "validation", "eda"] + (["upload"] if WANDB_TRACKING else [])
        result["status"] = ("PASS" if all(result[f"{key}_status"] == "PASS" for key in required) else
                            "PARTIAL" if result["training_status"] == "PASS" or result["stages"] else "FAIL")
        write_json(manifest_path, result)
        return result

## 04 — Jalankan fine-tuning dan tampilkan hasil

In [ ]:
from IPython.display import display, Image

RESULT = (fine_tuned_execute(RUN_DIR) if RAW_ARTIFACT_REF or RAW_MANIFEST_PATH else
          {"status": "NOT_RUN", "reason": "Isi RAW_ARTIFACT_REF atau RAW_MANIFEST_PATH setelah student tersedia."})
print(RESULT["status"], "|", RESULT.get("reason", RESULT.get("output", "")))
if RESULT.get("stages"):
    display(pd.DataFrame(fine_tuned_comparison(RESULT)[0]))
    for figure in sorted((RUN_DIR / "eda").glob("comparison_*.png")):
        display(Image(filename=str(figure)))